# Evaluation file


### Uses the exported results from the artifacts folder
to avoid rerunning entire notebooks

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

ART = Path("../artifacts/artifacts_raiders")

from utils.artifacts import load_artifacts
from utils.evaluation import (
    candidate_survival_rates,
    check_raw_occurrences,
    count_posts_with_term,
    extract_vocabulary,
    failure_stage,
    gram_to_string,
    identify_failure_stages,
    recover_vocabularies,
    track_terms,
)

(
    data,
    tfidf_candidates,
    candidate_scores,
    candidate_post_scores,
    candidate_metadata,
    anchor_metadata,
) = load_artifacts(ART)

### Define known expressions

cf Table IV of Kikkisetti et al. (2024)


In [4]:
known_terms = [
    "white genocide",
    "deep state",
    "interest groups",
    "kazarian mafia",
]

new_terms = [
    "fema camps",
    "klaus schwab",
    "central bank",
    "national socialist",
]

false_positives = [
    "plain sight",
    "german people",
    "new york city",
    "big part",
]

gold = pd.DataFrame({
    "term": known_terms + new_terms + false_positives,
    "label": (
        ["known"] * len(known_terms)
        + ["new"] * len(new_terms)
        + ["negative"] * len(false_positives)
    ),
})

display(gold)

,term,label
0,white genocide,known
1,deep state,known
2,interest groups,known
3,kazarian mafia,known
4,fema camps,new
5,klaus schwab,new
6,central bank,new
7,national socialist,new
8,plain sight,negative
9,german people,negative


In [8]:
ranking = candidate_scores.sort_values(
    "relevance", ascending=False
).reset_index(drop=True)

display(ranking.head(50))

,candidate,relevance
0,holocaust,0.866271
1,holocaust christian holocaust,0.859916
2,holocaust happened,0.845704
3,kike,0.832530
4,germany holocaust,0.832016
5,generations holocaust,0.828941
6,lolohoax,0.827880
7,holocaust narrative,0.824386
8,kike god,0.823677
9,kike lover,0.822876


In [9]:
ranking_counts = pd.read_csv(
    ART / "ranked_candidates_with_counts.csv",
    keep_default_na=False,
)

display(ranking_counts.head(50))

,candidate,relevance,n_posts,n_occurrences
0,holocaust,0.866271,287,349
1,holocaust christian holocaust,0.859916,1,1
2,holocaust happened,0.845704,1,1
3,kike,0.832530,501,520
4,germany holocaust,0.832016,1,1
5,generations holocaust,0.828941,2,2
6,lolohoax,0.827880,1,1
7,holocaust narrative,0.824386,2,2
8,kike god,0.823677,1,1
9,kike lover,0.822876,3,3


### recover the vocabulary at each stage

In [5]:
(
    vocab_before_pos,
    vocab_after_pos,
    vocab_after_tfidf,
    vocab_scored,
) = recover_vocabularies(data, tfidf_candidates, candidate_scores)

Before POS: 51801
After POS: 11587
After TF-IDF: 4661
Scored candidates: 4661


#### Check if expression exists in the raw corpus

In [6]:
gold = check_raw_occurrences(gold, data)

display(gold)

,term,label,raw_posts,in_raw
0,white genocide,known,2,True
1,deep state,known,0,False
2,interest groups,known,0,False
3,kazarian mafia,known,0,False
4,fema camps,new,0,False
5,klaus schwab,new,0,False
6,central bank,new,2,True
7,national socialist,new,1,True
8,plain sight,negative,0,False
9,german people,negative,1,True


### trace every expression through the pipeline

In [7]:
(
    gold,
    ranking,
    rank_lookup,
    score_lookup,
    threshold,
) = track_terms(
    gold, vocab_before_pos, vocab_after_pos, vocab_after_tfidf,
    vocab_scored, candidate_scores,
)

display(gold)

,term,label,raw_posts,in_raw,generated,after_pos,after_tfidf,scored,rank,relevance,passes_hmapp
0,white genocide,known,2,True,True,True,False,False,NaN,NaN,False
1,deep state,known,0,False,False,False,False,False,NaN,NaN,False
2,interest groups,known,0,False,False,False,False,False,NaN,NaN,False
3,kazarian mafia,known,0,False,False,False,False,False,NaN,NaN,False
4,fema camps,new,0,False,False,False,False,False,NaN,NaN,False
5,klaus schwab,new,0,False,False,False,False,False,NaN,NaN,False
6,central bank,new,2,True,True,True,True,True,3322.0,0.47462,False
7,national socialist,new,1,True,True,True,False,False,NaN,NaN,False
8,plain sight,negative,0,False,False,False,False,False,NaN,NaN,False
9,german people,negative,1,True,True,True,False,False,NaN,NaN,False


### identify the failure stage

In [ ]:
gold = identify_failure_stages(gold)

display(
    gold[[
        "term",
        "label",
        "raw_posts",
        "outcome",
        "rank",
        "relevance",
    ]]
)

,term,label,raw_posts,outcome,rank,relevance
0,white genocide,known,5,Retained by HMApp,791.0,0.617182
1,deep state,known,12,Retained by HMApp,2967.0,0.533629
2,interest groups,known,3,Semantic relevance cutoff,4966.0,0.483584
3,kazarian mafia,known,0,Absent from raw corpus,NaN,NaN
4,fema camps,new,2,Retained by HMApp,2881.0,0.536202
5,klaus schwab,new,9,Semantic relevance cutoff,4715.0,0.489486
6,central bank,new,3,Semantic relevance cutoff,6504.0,0.442916
7,national socialist,new,3,TF-IDF filtering,NaN,NaN
8,plain sight,negative,4,Semantic relevance cutoff,6210.0,0.451854
9,german people,negative,5,Retained by HMApp,1480.0,0.582787


## Eval metrics

### Candidate-stage recall

In [ ]:
positives = gold.loc[
    gold["label"].isin(["known", "new"])
    & gold["in_raw"]
]

stages = [
    "generated",
    "after_pos",
    "after_tfidf",
    "scored",
    "passes_hmapp",
]

survival_rates = candidate_survival_rates(positives, stages)

for stage, rate in survival_rates.items():
    print(f"{stage}: {rate:.1%}")

generated: 100.0%
after_pos: 100.0%
after_tfidf: 85.7%
scored: 85.7%
passes_hmapp: 42.9%


### Ranking quality

In [ ]:
display(
    gold.loc[
        gold["label"].isin(["known", "new"]),
        ["term", "rank", "relevance"]
    ].sort_values("rank")
)

,term,rank,relevance
0,white genocide,791.0,0.617182
4,fema camps,2881.0,0.536202
1,deep state,2967.0,0.533629
5,klaus schwab,4715.0,0.489486
2,interest groups,4966.0,0.483584
6,central bank,6504.0,0.442916
3,kazarian mafia,NaN,NaN
7,national socialist,NaN,NaN
